# EXP-A + EXP-B — Classical Baselines (Kaggle)

**Purpose:** retrain the classical baselines on the *same dataset, same split, same protocol* as EXP-C/D/E/F.

**Why this notebook exists.** The original EXP-A/EXP-B runs used a 27,000-image EuroSAT copy while
EXP-C/D/E/F used `apollo2506/eurosat-dataset`, which contains **27,600** images (SeaLake = 3,600 not 3,000).
The two arms therefore had different train *and* test sets, so classical-vs-quantum comparison was invalid.

**What is fixed here**
1. Same `DATASET_SLUG` as EXP-C  → identical file list, identical splits, identical normalisation statistics.
2. All seeds run the same epoch budget (30). Previously EXP-A seed 42 ran 50 while seeds 2021/7 ran 30.
3. Dataset composition is now **recorded into the metrics file** instead of printed as a soft warning.
4. Hard assertion on test-split composition against the quantum arm's known counts.

**Runtime:** ~1.5–2 h total on a single T4 for 6 runs (3 seeds x 2 experiments). No PennyLane needed.

**Before running:** Add Data -> attach `apollo2506/eurosat-dataset`. Accelerator -> GPU T4.

---
# Section 1 — Environment

In [1]:
# =============================================================================
# SECTION 1 — CELL 1.1: Kaggle Environment Detection + Path Setup
# =============================================================================
# MUST be identical to the quantum notebooks.
DATASET_SLUG: str = "datasets/apollo2506/eurosat-dataset"

# This notebook trains BOTH classical baselines.
EXPERIMENT_IDS: list = ["exp_a", "exp_b"]

# True  -> ignore completed.json / existing results and retrain from scratch.
# Keep True for the corrective re-run so stale 50-epoch EXP-A results cannot leak in.
FORCE_RETRAIN: bool = True

# Set True ONLY if you deliberately attached a previous run's checkpoints to resume.
RECOVER_CHECKPOINTS: bool = False
# -----------------------------------------------------------------------------

import os, sys, platform
import torch

print("=" * 70)
print("SECTION 1 — KAGGLE ENVIRONMENT DETECTION")
print("=" * 70)

assert os.path.isdir("/kaggle"), "ERROR: /kaggle not found. Run this on Kaggle."

KAGGLE_INPUT   = "/kaggle/input"
KAGGLE_WORKING = "/kaggle/working"
PROJECT_ROOT   = os.path.join(KAGGLE_WORKING, "Quantum-Geoscience")

print(f"\n[PATHS]")
print(f"  PROJECT_ROOT = {PROJECT_ROOT}")
print(f"  KAGGLE_INPUT = {KAGGLE_INPUT}")

_ds_path = os.path.join(KAGGLE_INPUT, DATASET_SLUG)
if not os.path.isdir(_ds_path):
    _avail = os.listdir(KAGGLE_INPUT) if os.path.isdir(KAGGLE_INPUT) else []
    raise FileNotFoundError(
        f"Dataset slug '{DATASET_SLUG}' not found at {_ds_path}.\n"
        f"Available in /kaggle/input: {_avail}\n"
        "FIX: Add Data -> attach apollo2506/eurosat-dataset.")
print(f"\n[DATASET] Found: {_ds_path}")

print(f"\n[RUNTIME]")
print(f"  Python : {sys.version.split()[0]}")
print(f"  torch  : {torch.__version__}")
if torch.cuda.is_available():
    for _gi in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_gi)
        print(f"  GPU {_gi}  : {_p.name}  ({_p.total_memory/1e9:.1f} GB)")
    print(f"  CUDA   : {torch.version.cuda}")
else:
    print("  WARNING: No GPU. Settings -> Accelerator -> GPU T4 and restart.")
print(f"\n[MODE] EXPERIMENT_IDS={EXPERIMENT_IDS}  FORCE_RETRAIN={FORCE_RETRAIN}")
print("[OK] Environment ready.")

SECTION 1 — KAGGLE ENVIRONMENT DETECTION

[PATHS]
  PROJECT_ROOT = /kaggle/working/Quantum-Geoscience
  KAGGLE_INPUT = /kaggle/input

[DATASET] Found: /kaggle/input/datasets/apollo2506/eurosat-dataset

[RUNTIME]
  Python : 3.12.13
  torch  : 2.10.0+cu128
  GPU 0  : Tesla T4  (15.6 GB)
  GPU 1  : Tesla T4  (15.6 GB)
  CUDA   : 12.8

[MODE] EXPERIMENT_IDS=['exp_a', 'exp_b']  FORCE_RETRAIN=True
[OK] Environment ready.


In [2]:
# =============================================================================
# SECTION 1 — CELL 1.2: pip helper utilities   (identical to EXP-C)
# =============================================================================
import subprocess, sys

def pip_install(*packages: str, check: bool = True) -> bool:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        if check:
            print(r.stderr[-3000:]); raise RuntimeError(f'pip install failed: {packages}')
        return False
    return True

def save_json(obj: dict, path: str) -> None:
    import json as _json, os, numpy as _np
    os.makedirs(os.path.dirname(path), exist_ok=True)
    class _Enc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, _np.ndarray):  return o.tolist()
            if isinstance(o, _np.integer):  return int(o)
            if isinstance(o, _np.floating): return float(o)
            if isinstance(o, tuple):        return list(o)
            return super().default(o)
    with open(path, 'w') as f:
        _json.dump(obj, f, cls=_Enc, indent=2)

print("pip_install / save_json defined.")

pip_install / save_json defined.


In [3]:
# =============================================================================
# SECTION 1 — CELL 1.3: Package Installation
# =============================================================================
# EXP-A/EXP-B are purely classical: NO PennyLane, NO autoray pin needed.
# We install only what the evaluation stack requires, at the SAME versions as EXP-C
# so that ECE / metrics are computed identically.
# =============================================================================
print("=" * 70)
print("SECTION 1 — PACKAGE INSTALLATION")
print("=" * 70)

import torch, torchvision, numpy as np
print(f"\n[PRE-INSTALLED — not reinstalled]")
print(f"  torch       {torch.__version__}")
print(f"  torchvision {torchvision.__version__}")
print(f"  numpy       {np.__version__}")

print("\n[installing] evaluation + data I/O ...")
for _p, _v in [("torchmetrics", "0.11.4"), ("einops", "0.6.0"), ("tifffile", "2024.2.12")]:
    pip_install(f"{_p}=={_v}")
    print(f"  {_p}=={_v}")

print("\n" + "=" * 70)
print("INSTALLATION COMPLETE")
print("=" * 70)

SECTION 1 — PACKAGE INSTALLATION

[PRE-INSTALLED — not reinstalled]
  torch       2.10.0+cu128
  torchvision 0.25.0+cu128
  numpy       2.0.2

[installing] evaluation + data I/O ...
  torchmetrics==0.11.4
  einops==0.6.0
  tifffile==2024.2.12

INSTALLATION COMPLETE


In [4]:
# =============================================================================
# SECTION 1 — CELL 1.4: Version Verification
# =============================================================================
import sys, torch, torchvision, torchmetrics
import numpy as np, scipy, pandas as pd, sklearn
import tifffile, einops

print("=" * 70)
print("SECTION 1 — VERSION VERIFICATION")
print("=" * 70)

_fail = []
def _chk_exact(name, inst, req):
    ok = inst == req
    print(f"  {'OK ' if ok else 'BAD'}  {name:<20} {inst:<16} (need {req})")
    if not ok: _fail.append(name)
def _chk_prefix(name, inst, pref, label=None):
    ok = inst.startswith(pref)
    print(f"  {'OK ' if ok else 'BAD'}  {name:<20} {inst:<16} (need {label or pref+'x'})")
    if not ok: _fail.append(name)

print("\n--- Python ---")
assert sys.version_info >= (3, 10)
print(f"  OK   Python {sys.version.split()[0]}")

print("\n--- Kaggle-native ---")
_chk_prefix("torch",        torch.__version__,       "2.", "torch 2.x")
_chk_prefix("torchvision",  torchvision.__version__, "0.", "torchvision 0.x")
_chk_prefix("numpy",        np.__version__,          "2.", "numpy 2.x")
_chk_prefix("scipy",        scipy.__version__,       "1.", "scipy 1.x")
_chk_prefix("pandas",       pd.__version__,          "2.", "pandas 2.x")
_chk_prefix("scikit-learn", sklearn.__version__,     "1.", "sklearn 1.x")

print("\n--- Installed here ---")
_chk_exact("torchmetrics", torchmetrics.__version__, "0.11.4")
_chk_exact("einops",       einops.__version__,       "0.6.0")
_chk_exact("tifffile",     tifffile.__version__,     "2024.2.12")

print("\n--- GPU ---")
if torch.cuda.is_available():
    for _gi in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_gi)
        print(f"  OK   GPU {_gi}: {_p.name} ({_p.total_memory/1e9:.1f} GB)")
else:
    print("  BAD  No GPU detected"); _fail.append("gpu")

print("\n" + "=" * 70)
if _fail:
    raise AssertionError(f"FAILED: {_fail}")
print("ALL CHECKS PASSED")
print("=" * 70)

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


SECTION 1 — VERSION VERIFICATION

--- Python ---
  OK   Python 3.12.13

--- Kaggle-native ---
  OK   torch                2.10.0+cu128     (need torch 2.x)
  OK   torchvision          0.25.0+cu128     (need torchvision 0.x)
  OK   numpy                2.0.2            (need numpy 2.x)
  OK   scipy                1.16.3           (need scipy 1.x)
  OK   pandas               2.3.3            (need pandas 2.x)
  OK   scikit-learn         1.6.1            (need sklearn 1.x)

--- Installed here ---
  OK   torchmetrics         0.11.4           (need 0.11.4)
  OK   einops               0.6.0            (need 0.6.0)
  OK   tifffile             2024.2.12        (need 2024.2.12)

--- GPU ---
  OK   GPU 0: Tesla T4 (15.6 GB)
  OK   GPU 1: Tesla T4 (15.6 GB)

ALL CHECKS PASSED


In [5]:
# =============================================================================
# SECTION 1 — CELL 1.5: Working Directory Tree
# =============================================================================
import os, shutil

_dirs = [
    os.path.join(PROJECT_ROOT, "data"),
    os.path.join(PROJECT_ROOT, "data", "splits"),
    os.path.join(PROJECT_ROOT, "data", "stats"),
    os.path.join(PROJECT_ROOT, "checkpoints"),
    os.path.join(PROJECT_ROOT, "figures"),
    os.path.join(PROJECT_ROOT, "tables"),
    os.path.join(PROJECT_ROOT, "exports"),
    os.path.join(PROJECT_ROOT, "logs"),
]
for _e in EXPERIMENT_IDS:
    _dirs.append(os.path.join(PROJECT_ROOT, "checkpoints", _e))
_dirs.append(os.path.join(PROJECT_ROOT, "checkpoints", "sanity"))  # used by Section 6.4 sanity check
for d in _dirs:
    os.makedirs(d, exist_ok=True)

# Checkpoint recovery is DISABLED by default for this corrective re-run.
# Stale EXP-A checkpoints (the 50-epoch seed-42 run) would otherwise be resumed
# and silently reproduce the mismatched protocol we are trying to fix.
if RECOVER_CHECKPOINTS:
    _ckpt_dst = os.path.join(PROJECT_ROOT, "checkpoints")
    for _inp_name in os.listdir("/kaggle/input"):
        _inp_ckpt = os.path.join("/kaggle/input", _inp_name, "Quantum-Geoscience", "checkpoints")
        if os.path.isdir(_inp_ckpt):
            print(f"  Recovering checkpoints from /kaggle/input/{_inp_name}/ ...")
            for _exp in os.listdir(_inp_ckpt):
                if _exp not in EXPERIMENT_IDS:
                    continue
                _src_exp = os.path.join(_inp_ckpt, _exp)
                _dst_exp = os.path.join(_ckpt_dst, _exp)
                os.makedirs(_dst_exp, exist_ok=True)
                for _f in os.listdir(_src_exp):
                    _sf, _df = os.path.join(_src_exp, _f), os.path.join(_dst_exp, _f)
                    if not os.path.exists(_df):
                        shutil.copy2(_sf, _df); print(f"    Recovered: {_exp}/{_f}")
else:
    print("  RECOVER_CHECKPOINTS=False — starting clean (recommended for this re-run).")

print(f"Directory tree ready under {PROJECT_ROOT}")
print(f"  Checkpoint dirs: {[os.path.join('checkpoints', e) for e in EXPERIMENT_IDS]}")
print(f"  Sanity checkpoint dir: checkpoints/sanity  (exists={os.path.isdir(os.path.join(PROJECT_ROOT,'checkpoints','sanity'))})")

  RECOVER_CHECKPOINTS=False — starting clean (recommended for this re-run).
Directory tree ready under /kaggle/working/Quantum-Geoscience
  Checkpoint dirs: ['checkpoints/exp_a', 'checkpoints/exp_b']
  Sanity checkpoint dir: checkpoints/sanity  (exists=True)


---
# Section 2 — Configuration

In [6]:
# =============================================================================
# SECTION 2 — CELL 2.1: Guard + Core Imports
# =============================================================================
assert 'PROJECT_ROOT' in globals(), 'Run Section 1 first.'

import os, sys, json, hashlib, random, math, gc, time
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.models as tvm
import torchmetrics

print('Core imports OK.')

Core imports OK.


In [7]:
# =============================================================================
# SECTION 2 — CELL 2.2: set_all_seeds   (identical to EXP-C)
# =============================================================================
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print('set_all_seeds() defined.')

set_all_seeds() defined.


In [8]:
# =============================================================================
# SECTION 2 — CELL 2.3: CONFIG
# =============================================================================
# Every shared key below is byte-identical to the EXP-C notebook so that the
# classical and quantum arms are trained and evaluated under one protocol.
# Quantum-only keys (n_qubits_*, vqc_*, diff_method, hessian_*, pennylane_device)
# are intentionally absent — they do not apply to EXP-A/EXP-B.
# =============================================================================
CONFIG: dict = {
    # PATHS
    "project_root":   PROJECT_ROOT,
    "data_dir":       os.path.join(PROJECT_ROOT, "data"),
    "eurosat_dir":    None,            # set in Cell 3.3
    "splits_dir":     os.path.join(PROJECT_ROOT, "data", "splits"),
    "stats_dir":      os.path.join(PROJECT_ROOT, "data", "stats"),
    "checkpoint_dir": os.path.join(PROJECT_ROOT, "checkpoints"),
    "figures_dir":    os.path.join(PROJECT_ROOT, "figures"),
    "tables_dir":     os.path.join(PROJECT_ROOT, "tables"),
    "exports_dir":    os.path.join(PROJECT_ROOT, "exports"),
    "logs_dir":       os.path.join(PROJECT_ROOT, "logs"),

    # DATASET (frozen — same as EXP-C)
    "image_size": 64, "n_classes": 10,
    "n_bands_rgb": 3, "n_bands_nir": 1,
    "reflectance_scale": 10_000.0,
    "split_seed": 42,
    "train_ratio": 0.70, "val_ratio": 0.15, "test_ratio": 0.15,

    # DATALOADER (same as EXP-C)
    "batch_size":         128,
    "num_workers":        4,
    "pin_memory":         True,
    "persistent_workers": True,
    "prefetch_factor":    2,

    # MODEL
    "backbone_output_dim": 64,
    "classifier_hidden":   128,

    # TRAINING (same as EXP-C — note max_epochs=30 for ALL seeds)
    "max_epochs":             30,
    "lr":                     0.001,
    "weight_decay_classical": 1e-5,
    "weight_decay_quantum":   0.0,
    "adam_betas":             (0.9, 0.999),
    "adam_eps":               1e-8,
    "lr_scheduler":           "CosineAnnealingLR",
    "cosine_T_max":           30,
    "cosine_eta_min":         1e-5,
    "grad_clip_norm":         1.0,
    "early_stop_patience":    7,
    "early_stop_min_delta":   0.001,

    # UNCERTAINTY (same as EXP-C)
    "n_ensemble":         3,
    "ensemble_seeds":     [42, 2021, 7],
    "n_independent_runs": 3,
    "ece_n_bins":         15,
    "ood_noise_levels":   [0.00, 0.01, 0.05, 0.10, 0.20],

    # STATISTICS
    "alpha":        0.05,
    "bonferroni_n": 6,
    "bootstrap_B":  10_000,

    # FIGURES
    "figure_dpi": 300, "figure_format": "pdf",

    # KAGGLE
    "dataset_slug":       DATASET_SLUG,
    "platform":           "kaggle",
    "kaggle_input_dir":   "/kaggle/input",
    "kaggle_working_dir": "/kaggle/working",
    "experiment_ids":     EXPERIMENT_IDS,
}

assert CONFIG["cosine_T_max"] == CONFIG["max_epochs"]
assert len(CONFIG["ensemble_seeds"]) == CONFIG["n_ensemble"]

print(f"CONFIG defined — {len(CONFIG)} keys.")
print(f"  experiments  = {CONFIG['experiment_ids']}")
print(f"  max_epochs   = {CONFIG['max_epochs']} (ALL seeds — this is the fix)")
print(f"  batch_size   = {CONFIG['batch_size']}")
print(f"  seeds        = {CONFIG['ensemble_seeds']}")

CONFIG defined — 53 keys.
  experiments  = ['exp_a', 'exp_b']
  max_epochs   = 30 (ALL seeds — this is the fix)
  batch_size   = 128
  seeds        = [42, 2021, 7]


In [9]:
# =============================================================================
# SECTION 2 — CELL 2.4: CONFIG hash + serialisation
# =============================================================================
import hashlib

def _config_hash(cfg: dict) -> str:
    import json as _j
    serial = _j.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in cfg.items()}, sort_keys=True)
    return hashlib.md5(serial.encode()).hexdigest()

CONFIG_HASH = _config_hash(CONFIG)
save_json({k: (list(v) if isinstance(v, tuple) else v) for k, v in CONFIG.items()},
          os.path.join(CONFIG['exports_dir'], 'configuration_ab.json'))
print(f'configuration_ab.json written. CONFIG_HASH = {CONFIG_HASH}')
print('NOTE: this hash differs from EXP-C by design (quantum keys absent).')
print('      The shared training/eval keys are identical — that is what matters.')

configuration_ab.json written. CONFIG_HASH = 9af35c188e63a728c411086147d92839
NOTE: this hash differs from EXP-C by design (quantum keys absent).
      The shared training/eval keys are identical — that is what matters.


---
# Section 3 — Dataset, Splits, Integrity Verification

In [10]:
# =============================================================================
# SECTION 3 — CELL 3.1: Guard + imports
# =============================================================================
assert "CONFIG" in globals(), "Run Section 2 first."
import os, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
import tifffile
from sklearn.model_selection import StratifiedShuffleSplit
print("Section 3 imports OK.")

Section 3 imports OK.


In [11]:
# =============================================================================
# SECTION 3 — CELL 3.2: EuroSATDataset   (byte-identical to EXP-C)
# =============================================================================
class EuroSATDataset(Dataset):
    """
    EuroSAT multispectral loader with RGB/NIR band separation.
    Reads 64x64 GeoTIFF patches (13 Sentinel-2 bands) and extracts
    B02/B03/B04 (RGB) and B08 (NIR); scales by 1/10000 then z-scores
    using training-split statistics.
    Returns: rgb (3,64,64), nir (1,64,64), label int.
    """
    CLASS_NAMES = [
        'AnnualCrop','Forest','HerbaceousVegetation','Highway',
        'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake',
    ]
    RGB_IDX = [1, 2, 3]
    NIR_IDX = [7]
    REFLECTANCE_SCALE = 10_000.0
    NUM_CLASSES = 10

    def __init__(self, root_dir: str, indices: np.ndarray,
                 mean: np.ndarray, std: np.ndarray, augment: bool = False):
        self.root_dir = root_dir
        self.mean = mean.astype(np.float32)
        self.std  = std.astype(np.float32)
        self.augment = augment

        all_files, all_labels = [], []
        for cls_idx, cls_name in enumerate(self.CLASS_NAMES):
            cls_dir = os.path.join(root_dir, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            for fname in sorted(os.listdir(cls_dir)):
                if fname.endswith('.tif'):
                    all_files.append(os.path.join(cls_dir, fname))
                    all_labels.append(cls_idx)

        self.files  = [all_files[i]  for i in indices]
        self.labels = [all_labels[i] for i in indices]

    def __len__(self) -> int:
        return len(self.files)

    def _load_bands(self, path: str):
        img = tifffile.imread(path)
        if img.ndim == 3:
            if img.shape[-1] >= 13:
                rgb = img[:, :, self.RGB_IDX].astype(np.float32)
                nir = img[:, :, self.NIR_IDX].astype(np.float32)
            else:
                rgb = img[self.RGB_IDX, :, :].astype(np.float32).transpose(1, 2, 0)
                nir = img[self.NIR_IDX, :, :].astype(np.float32).transpose(1, 2, 0)
        else:
            raise ValueError(f"Unexpected image shape {img.shape} in {path}")
        rgb /= self.REFLECTANCE_SCALE
        nir /= self.REFLECTANCE_SCALE
        return rgb, nir

    def __getitem__(self, idx: int):
        rgb_hw3, nir_hw1 = self._load_bands(self.files[idx])
        rgb_hw3 = (rgb_hw3 - self.mean[:3]) / (self.std[:3] + 1e-8)
        nir_hw1 = (nir_hw1 - self.mean[3:]) / (self.std[3:] + 1e-8)
        rgb = torch.from_numpy(rgb_hw3.transpose(2, 0, 1))
        nir = torch.from_numpy(nir_hw1.transpose(2, 0, 1))
        if self.augment:
            import torchvision.transforms.functional as TF
            if torch.rand(1) > 0.5: rgb = TF.hflip(rgb); nir = TF.hflip(nir)
            if torch.rand(1) > 0.5: rgb = TF.vflip(rgb); nir = TF.vflip(nir)
            angle = (torch.rand(1).item() - 0.5) * 30
            rgb = TF.rotate(rgb, angle); nir = TF.rotate(nir, angle)
        return rgb, nir, self.labels[idx]

print("EuroSATDataset defined.")

EuroSATDataset defined.


In [12]:
# =============================================================================
# SECTION 3 — CELL 3.3: EuroSAT auto-detection   (identical to EXP-C)
# =============================================================================
import os
_DS_ROOT = os.path.join("/kaggle/input", DATASET_SLUG)
_CLASSES = EuroSATDataset.CLASS_NAMES

def _count_class_dirs(d):
    if not os.path.isdir(d): return 0
    return sum(1 for n in _CLASSES if os.path.isdir(os.path.join(d, n)))

def _find_eurosat_root(root):
    if _count_class_dirs(root) >= 5:
        return root
    for _r, _dirs, _ in os.walk(root):
        if _r.replace(root, "").count(os.sep) > 6:
            _dirs.clear(); continue
        if _count_class_dirs(_r) >= 5:
            return _r
    return None

print(f"Searching for EuroSAT class dirs under {_DS_ROOT} ...")
print("  (Looking for EuroSATallBands/ — the 13-band multispectral version)")
EUROSAT_DIR = _find_eurosat_root(_DS_ROOT)
if EUROSAT_DIR is None:
    raise FileNotFoundError(
        f"Cannot find EuroSAT class directories under {_DS_ROOT}.\n"
        f"Contents: {os.listdir(_DS_ROOT) if os.path.isdir(_DS_ROOT) else 'MISSING'}")

CONFIG["eurosat_dir"] = EUROSAT_DIR
_found = [n for n in _CLASSES if os.path.isdir(os.path.join(EUROSAT_DIR, n))]
assert len(_found) == len(_CLASSES), f"Only {len(_found)}/10 class dirs found"
print(f"  EUROSAT_DIR = {EUROSAT_DIR}")
print(f"  Classes found: {len(_found)}/10 — dataset ready")

Searching for EuroSAT class dirs under /kaggle/input/datasets/apollo2506/eurosat-dataset ...
  (Looking for EuroSATallBands/ — the 13-band multispectral version)
  EUROSAT_DIR = /kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSATallBands
  Classes found: 10/10 — dataset ready


In [13]:
# =============================================================================
# SECTION 3 — CELL 3.4: Build file list + RECORD dataset composition
# =============================================================================
# This is the cell that previously printed a soft warning and continued.
# It now records the composition into DATASET_MANIFEST, which is written into
# the metrics file so the dataset variant is part of the permanent record.
# =============================================================================
_all_files, _all_labels = [], []
_per_class_counts = {}

for _cls_idx, _cls_name in enumerate(_CLASSES):
    _cls_dir = os.path.join(EUROSAT_DIR, _cls_name)
    _files = sorted([f for f in os.listdir(_cls_dir) if f.endswith(".tif")])
    _all_files  += [os.path.join(_cls_dir, f) for f in _files]
    _all_labels += [_cls_idx] * len(_files)
    _per_class_counts[_cls_name] = len(_files)
    print(f"  {_cls_name:<24} {len(_files):>6} images")

_all_labels_np = np.array(_all_labels, dtype=np.int64)
_total = len(_all_files)

# Canonical EuroSAT composition, for reference
_CANONICAL = {'AnnualCrop':3000,'Forest':3000,'HerbaceousVegetation':3000,
              'Highway':2500,'Industrial':2500,'Pasture':2000,
              'PermanentCrop':2500,'Residential':3000,'River':2500,'SeaLake':3000}

_deltas = {k: _per_class_counts[k] - _CANONICAL[k]
           for k in _CANONICAL if _per_class_counts[k] != _CANONICAL[k]}

DATASET_MANIFEST = {
    "dataset_slug":        DATASET_SLUG,
    "eurosat_dir":         EUROSAT_DIR,
    "total_images":        _total,
    "per_class_counts":    _per_class_counts,
    "canonical_total":     sum(_CANONICAL.values()),
    "deviations_from_canonical": _deltas,
    "is_canonical_eurosat": (_total == 27_000 and not _deltas),
}

print(f"\n  TOTAL: {_total} images")
if DATASET_MANIFEST["is_canonical_eurosat"]:
    print("  This IS the canonical 27,000-image EuroSAT.")
else:
    print(f"  This is NOT canonical EuroSAT (canonical = 27,000).")
    print(f"  Deviations: {_deltas}")
    print("  -> This variant MUST be declared in the paper's dataset section.")
    print("  -> Literature figures (e.g. Helber 98.57%) are NOT directly comparable.")
print("\n  Composition recorded in DATASET_MANIFEST (written to metrics file).")

  AnnualCrop                 3000 images
  Forest                     3000 images
  HerbaceousVegetation       3000 images
  Highway                    2500 images
  Industrial                 2500 images
  Pasture                    2000 images
  PermanentCrop              2500 images
  Residential                3000 images
  River                      2500 images
  SeaLake                    3597 images

  TOTAL: 27597 images
  This is NOT canonical EuroSAT (canonical = 27,000).
  Deviations: {'SeaLake': 597}
  -> This variant MUST be declared in the paper's dataset section.
  -> Literature figures (e.g. Helber 98.57%) are NOT directly comparable.

  Composition recorded in DATASET_MANIFEST (written to metrics file).


In [14]:
# =============================================================================
# SECTION 3 — CELL 3.5: Generate stratified splits   (identical logic to EXP-C)
# =============================================================================
# Same dataset + same code + random_state=42 => bit-identical indices to the
# quantum arm. Cell 3.5b then verifies this against the known quantum split.
# =============================================================================
SPLITS_DIR  = CONFIG["splits_dir"]
_train_path = os.path.join(SPLITS_DIR, "train_indices.npy")
_val_path   = os.path.join(SPLITS_DIR, "val_indices.npy")
_test_path  = os.path.join(SPLITS_DIR, "test_indices.npy")

if not (os.path.exists(_train_path) and os.path.exists(_val_path) and os.path.exists(_test_path)):
    print("Generating stratified splits (seed 42)...")
    _sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    _train_idx, _holdout_idx = next(_sss1.split(np.zeros(len(_all_files)), _all_labels_np))
    _sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    _val_idx_rel, _test_idx_rel = next(_sss2.split(
        np.zeros(len(_holdout_idx)), _all_labels_np[_holdout_idx]))
    _val_idx  = _holdout_idx[_val_idx_rel]
    _test_idx = _holdout_idx[_test_idx_rel]
    np.save(_train_path, _train_idx); np.save(_val_path, _val_idx); np.save(_test_path, _test_idx)
    print("  Splits saved.")
else:
    print("Loading existing splits...")
    _train_idx = np.load(_train_path)
    _val_idx   = np.load(_val_path)
    _test_idx  = np.load(_test_path)

CONFIG["n_train"] = len(_train_idx)
CONFIG["n_val"]   = len(_val_idx)
CONFIG["n_test"]  = len(_test_idx)

assert len(set(_train_idx) & set(_val_idx))  == 0, "train/val overlap"
assert len(set(_train_idx) & set(_test_idx)) == 0, "train/test overlap"
assert len(set(_val_idx)   & set(_test_idx)) == 0, "val/test overlap"

print(f"  train={len(_train_idx)}  val={len(_val_idx)}  test={len(_test_idx)}")
print(f"  total={len(_train_idx)+len(_val_idx)+len(_test_idx)}  (no overlaps)")

Generating stratified splits (seed 42)...
  Splits saved.
  train=19317  val=4140  test=4140
  total=27597  (no overlaps)


In [15]:
# =============================================================================
# SECTION 3 — CELL 3.5b: *** SPLIT MATCH VERIFICATION vs QUANTUM ARM ***
# =============================================================================
# The whole point of this notebook. EXP-C/D/E/F were evaluated on a test split
# with this exact per-class composition. If EXP-A/B do not match it here, the
# classical-vs-quantum comparison is invalid and training must not proceed.
# =============================================================================
QUANTUM_TEST_PER_CLASS = [450, 450, 450, 375, 375, 300, 375, 450, 375, 540]
QUANTUM_TEST_TOTAL     = 4140

_test_counts = np.bincount(_all_labels_np[_test_idx], minlength=10).tolist()
_val_counts  = np.bincount(_all_labels_np[_val_idx],  minlength=10).tolist()

print("Test-split composition (this run vs quantum arm):")
print(f"  {'class':<24}{'this run':>10}{'quantum':>10}{'match':>8}")
_ok = True
for _i, _n in enumerate(_CLASSES):
    _a, _b = _test_counts[_i], QUANTUM_TEST_PER_CLASS[_i]
    _m = (_a == _b); _ok &= _m
    print(f"  {_n:<24}{_a:>10}{_b:>10}{'yes' if _m else 'NO':>8}")
print(f"  {'TOTAL':<24}{sum(_test_counts):>10}{QUANTUM_TEST_TOTAL:>10}"
      f"{'yes' if sum(_test_counts)==QUANTUM_TEST_TOTAL else 'NO':>8}")

# Fingerprint of the actual index arrays — record this and compare across notebooks
import hashlib
def _idx_md5(a): return hashlib.md5(np.ascontiguousarray(np.sort(a)).tobytes()).hexdigest()[:16]
SPLIT_FINGERPRINT = {
    "train_md5": _idx_md5(_train_idx),
    "val_md5":   _idx_md5(_val_idx),
    "test_md5":  _idx_md5(_test_idx),
    "test_per_class": _test_counts,
    "val_per_class":  _val_counts,
}
print(f"\n  SPLIT FINGERPRINT")
for _k, _v in SPLIT_FINGERPRINT.items():
    if _k.endswith('_md5'): print(f"    {_k:<12} {_v}")

if not _ok:
    raise AssertionError(
        "SPLIT MISMATCH — this run's test split does NOT match the quantum arm.\n"
        f"  this run: {_test_counts}\n"
        f"  quantum : {QUANTUM_TEST_PER_CLASS}\n"
        "Most likely cause: a different EuroSAT copy is attached.\n"
        f"Check DATASET_SLUG (currently {DATASET_SLUG!r}) matches the quantum notebooks.")

print("\n  SPLIT MATCHES THE QUANTUM ARM — safe to proceed.")

Test-split composition (this run vs quantum arm):
  class                     this run   quantum   match
  AnnualCrop                     450       450     yes
  Forest                         450       450     yes
  HerbaceousVegetation           450       450     yes
  Highway                        375       375     yes
  Industrial                     375       375     yes
  Pasture                        300       300     yes
  PermanentCrop                  375       375     yes
  Residential                    450       450     yes
  River                          375       375     yes
  SeaLake                        540       540     yes
  TOTAL                         4140      4140     yes

  SPLIT FINGERPRINT
    train_md5    c52c6de601dfc9f5
    val_md5      58f63e10e0367518
    test_md5     5d5ff68d7e75b127

  SPLIT MATCHES THE QUANTUM ARM — safe to proceed.


In [16]:
# =============================================================================
# SECTION 3 — CELL 3.5c: SeaLake duplicate check  (diagnostic, ~1 min)
# =============================================================================
# The non-canonical class. If unique < file count, the surplus images are exact
# duplicates and copies may straddle train/test. Recorded either way.
# =============================================================================
RUN_DUPLICATE_CHECK: bool = True

DUPLICATE_REPORT = {"ran": False}
if RUN_DUPLICATE_CHECK:
    import hashlib, collections
    _cls = "SeaLake"
    _cdir = os.path.join(EUROSAT_DIR, _cls)
    _cfiles = sorted(f for f in os.listdir(_cdir) if f.endswith(".tif"))
    print(f"Hashing {len(_cfiles)} {_cls} images ...")

    _h = collections.defaultdict(list)
    for _f in _cfiles:
        _arr = tifffile.imread(os.path.join(_cdir, _f))
        _h[hashlib.md5(np.ascontiguousarray(_arr)).hexdigest()].append(_f)

    _dupes = {k: v for k, v in _h.items() if len(v) > 1}
    _redundant = sum(len(v) - 1 for v in _dupes.values())

    # Do duplicate copies straddle the train/test boundary?
    _name2split = {}
    for _split_name, _idx in [("train", _train_idx), ("val", _val_idx), ("test", _test_idx)]:
        for _i in _idx:
            if _all_labels_np[_i] == _CLASSES.index(_cls):
                _name2split[os.path.basename(_all_files[_i])] = _split_name
    _leaky = 0
    for _grp in _dupes.values():
        _splits = {_name2split.get(g) for g in _grp} - {None}
        if len(_splits) > 1 and ("train" in _splits) and ({"test", "val"} & _splits):
            _leaky += 1

    DUPLICATE_REPORT = {
        "ran": True, "class": _cls,
        "file_count": len(_cfiles), "unique_images": len(_h),
        "duplicate_groups": len(_dupes), "redundant_files": _redundant,
        "groups_straddling_train_and_heldout": _leaky,
    }
    print(f"  files={len(_cfiles)}  unique={len(_h)}  "
          f"dup_groups={len(_dupes)}  redundant={_redundant}")
    if _redundant == 0:
        print("  No duplicates — the surplus images are distinct scenes.")
        print("  -> No leakage. Only the non-canonical composition must be declared.")
    else:
        print(f"  DUPLICATES FOUND. Groups spanning train and held-out: {_leaky}")
        print("  -> Declare this in the paper. Note SeaLake F1 is ~0.99 in every")
        print("     experiment, so measurable OA impact is near zero, but it must be stated.")
else:
    print("Duplicate check skipped (RUN_DUPLICATE_CHECK=False).")

Hashing 3597 SeaLake images ...
  files=3597  unique=3597  dup_groups=0  redundant=0
  No duplicates — the surplus images are distinct scenes.
  -> No leakage. Only the non-canonical composition must be declared.


In [17]:
# =============================================================================
# SECTION 3 — CELL 3.6: Normalisation statistics   (identical to EXP-C)
# =============================================================================
STATS_DIR  = CONFIG["stats_dir"]
_mean_path = os.path.join(STATS_DIR, "channel_mean.npy")
_std_path  = os.path.join(STATS_DIR, "channel_std.npy")

if not (os.path.exists(_mean_path) and os.path.exists(_std_path)):
    print("Computing channel statistics from 5,000 training-split samples...")
    np.random.seed(42)
    _sample_idx = np.random.choice(_train_idx, size=min(5000, len(_train_idx)), replace=False)
    _sum4, _sum2_4, _N = np.zeros(4), np.zeros(4), 0
    for _i, _idx in enumerate(_sample_idx):
        _img = tifffile.imread(_all_files[_idx]).astype(np.float64) / 10_000.0
        if _img.ndim == 3 and _img.shape[-1] >= 13:
            _bands = np.stack([_img[:,:,1], _img[:,:,2], _img[:,:,3], _img[:,:,7]], axis=-1)
        else:
            _bands = np.stack([_img[1,:,:], _img[2,:,:], _img[3,:,:], _img[7,:,:]], axis=-1)
        _sum4   += _bands.reshape(-1, 4).mean(0)
        _sum2_4 += (_bands.reshape(-1, 4) ** 2).mean(0)
        _N += 1
        if (_i + 1) % 1000 == 0: print(f"  {_i+1}/{len(_sample_idx)}")
    _mean = (_sum4 / _N).astype(np.float32)
    _std  = np.sqrt(np.maximum(_sum2_4/_N - (_sum4/_N)**2, 1e-12)).astype(np.float32)
    np.save(_mean_path, _mean); np.save(_std_path, _std)
    print("  Statistics saved.")
else:
    _mean = np.load(_mean_path); _std = np.load(_std_path)
    print("Loaded existing statistics.")

assert _mean.shape == (4,) and _std.shape == (4,)
assert np.isfinite(_mean).all() and np.isfinite(_std).all()
CHANNEL_MEAN, CHANNEL_STD = _mean, _std
STATS_FINGERPRINT = {"mean": _mean.round(6).tolist(), "std": _std.round(6).tolist()}
print(f"  mean = {_mean.round(4)}")
print(f"  std  = {_std.round(4)}")
print("  (compare these against the EXP-C run — they must be identical)")

Computing channel statistics from 5,000 training-split samples...
  1000/5000
  2000/5000
  3000/5000
  4000/5000
  5000/5000
  Statistics saved.
  mean = [0.1113 0.1031 0.0933 0.2243]
  std  = [0.0329 0.0395 0.0592 0.1148]
  (compare these against the EXP-C run — they must be identical)


In [18]:
# =============================================================================
# SECTION 3 — CELL 3.7: Datasets + DataLoaders   (identical to EXP-C)
# =============================================================================
train_dataset = EuroSATDataset(EUROSAT_DIR, _train_idx, CHANNEL_MEAN, CHANNEL_STD, augment=True)
val_dataset   = EuroSATDataset(EUROSAT_DIR, _val_idx,   CHANNEL_MEAN, CHANNEL_STD, augment=False)
test_dataset  = EuroSATDataset(EUROSAT_DIR, _test_idx,  CHANNEL_MEAN, CHANNEL_STD, augment=False)

_loader_kw = dict(
    num_workers        = CONFIG["num_workers"],
    pin_memory         = CONFIG["pin_memory"],
    persistent_workers = CONFIG["persistent_workers"],
    prefetch_factor    = CONFIG["prefetch_factor"],
    drop_last          = False,
)
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  **_loader_kw)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, **_loader_kw)

_rgb, _nir, _lbl = next(iter(train_loader))
assert _rgb.shape == (CONFIG["batch_size"], 3, 64, 64)
assert _nir.shape == (CONFIG["batch_size"], 1, 64, 64)
assert not torch.isnan(_rgb).any() and not torch.isnan(_nir).any()

print("Section 3 DataLoaders PASSED:")
print(f"  train : {len(train_loader)} batches x {CONFIG['batch_size']}  ({len(train_dataset)} imgs)")
print(f"  val   : {len(val_loader)} batches  ({len(val_dataset)} imgs)")
print(f"  test  : {len(test_loader)} batches  ({len(test_dataset)} imgs)")

Section 3 DataLoaders PASSED:
  train : 151 batches x 128  (19317 imgs)
  val   : 33 batches  (4140 imgs)
  test  : 33 batches  (4140 imgs)


---
# Section 4 — Models (EXP-A, EXP-B)

In [19]:
# =============================================================================
# SECTION 4 — CELL 4.1: EXP-A / EXP-B classifiers   (byte-identical to working nb)
# =============================================================================
import torchvision.models as tvm

class ClassicalRGBClassifier(nn.Module):
    """EXP-A: classical RGB-only baseline. ResNet-18 (3ch, stride-1 conv1, no maxpool) -> Linear(512,10)."""
    def __init__(self):
        super().__init__()
        _base = tvm.resnet18(weights=None)
        _base.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        _base.maxpool = nn.Identity()
        _base.fc      = nn.Linear(512, CONFIG["n_classes"])
        self.model = _base

    def forward(self, rgb: torch.Tensor, nir: torch.Tensor = None) -> torch.Tensor:
        return self.model(rgb)


class ClassicalRGBNIRClassifier(nn.Module):
    """EXP-B: classical 4-channel (RGB+NIR) baseline."""
    def __init__(self):
        super().__init__()
        _base = tvm.resnet18(weights=None)
        _base.conv1   = nn.Conv2d(4, 64, kernel_size=3, stride=1, padding=1, bias=False)
        _base.maxpool = nn.Identity()
        _base.fc      = nn.Linear(512, CONFIG["n_classes"])
        self.model = _base

    def forward(self, rgb: torch.Tensor, nir: torch.Tensor) -> torch.Tensor:
        return self.model(torch.cat([rgb, nir], dim=1))


EXP_MODEL_MAP = {"exp_a": ClassicalRGBClassifier, "exp_b": ClassicalRGBNIRClassifier}

def build_model(exp_id: str) -> nn.Module:
    return EXP_MODEL_MAP[exp_id]().cuda()

# ── Verification ─────────────────────────────────────────────────────────────
_r = torch.randn(2, 3, 64, 64).cuda(); _n = torch.randn(2, 1, 64, 64).cuda()
_mA, _mB = build_model("exp_a"), build_model("exp_b")
assert _mA(_r, _n).shape == (2, 10)
assert _mB(_r, _n).shape == (2, 10)
_nA = sum(p.numel() for p in _mA.parameters())
_nB = sum(p.numel() for p in _mB.parameters())
print(f"EXP-A params: {_nA:,}   (previous run: 11,173,962)")
print(f"EXP-B params: {_nB:,}   (previous run: 11,174,538)")
assert _nA == 11_173_962, "EXP-A architecture changed — must match previous run"
assert _nB == 11_174_538, "EXP-B architecture changed — must match previous run"
del _mA, _mB; torch.cuda.empty_cache(); gc.collect()
print("Section 4 PASSED — architectures identical to the original runs.")

EXP-A params: 11,173,962   (previous run: 11,173,962)
EXP-B params: 11,174,538   (previous run: 11,174,538)
Section 4 PASSED — architectures identical to the original runs.


---
# Section 5 — Training Machinery

In [20]:
# =============================================================================
# SECTION 5 — CELL 5.1: Optimizer + Scheduler
# =============================================================================
# EXP-C splits params into classical (wd=1e-5) and quantum (wd=0.0) groups.
# EXP-A/B have no quantum params, so the quantum group is empty and this reduces
# to a single group at wd=1e-5 — mathematically identical to the EXP-C path.
# =============================================================================
def build_optimizer(model: nn.Module) -> torch.optim.AdamW:
    classical = list(model.parameters())
    return torch.optim.AdamW(
        [{"params": classical, "weight_decay": CONFIG["weight_decay_classical"]}],
        lr=CONFIG["lr"], betas=CONFIG["adam_betas"], eps=CONFIG["adam_eps"])


def build_scheduler(optimizer) -> torch.optim.lr_scheduler.CosineAnnealingLR:
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["cosine_T_max"], eta_min=CONFIG["cosine_eta_min"])


class EarlyStopping:
    def __init__(self, patience: int, min_delta: float):
        self.patience = patience; self.min_delta = min_delta
        self.best_loss = float("inf"); self.counter = 0; self.should_stop = False

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

print("build_optimizer / build_scheduler / EarlyStopping defined.")

build_optimizer / build_scheduler / EarlyStopping defined.


In [21]:
# =============================================================================
# SECTION 5 — CELL 5.2: Checkpoint save / load   (identical to EXP-C)
# =============================================================================
def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, val_acc,
                    seed, exp_id, training_log, is_best=False) -> str:
    suffix = "_best" if is_best else ""
    name   = f"{exp_id}_seed{seed}_epoch{epoch:03d}_valAcc{val_acc:.4f}{suffix}.pt"
    path   = os.path.join(CONFIG["checkpoint_dir"], exp_id, name)
    _state = (model.module if isinstance(model, torch.nn.DataParallel) else model).state_dict()
    torch.save({
        "epoch": epoch, "model_state_dict": _state,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_loss": val_loss, "val_acc": val_acc,
        "seed": seed, "exp_id": exp_id, "config_hash": CONFIG_HASH,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "training_log": training_log,
    }, path)
    return path


def load_checkpoint(model, path, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cuda", weights_only=False)
    state = ckpt["model_state_dict"]
    if list(state.keys())[0].startswith("module."):
        state = {k[7:]: v for k, v in state.items()}
    _m = model.module if isinstance(model, torch.nn.DataParallel) else model
    _m.load_state_dict(state)
    if optimizer: optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler: scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    return ckpt


def get_best_checkpoint(exp_id: str, seed: int):
    d = os.path.join(CONFIG["checkpoint_dir"], exp_id)
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.startswith(f"{exp_id}_seed{seed}_") and "_best" in f]
    return os.path.join(d, sorted(c)[-1]) if c else None


def get_latest_checkpoint(exp_id: str, seed: int):
    d = os.path.join(CONFIG["checkpoint_dir"], exp_id)
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.startswith(f"{exp_id}_seed{seed}_") and f.endswith(".pt")]
    return os.path.join(d, sorted(c)[-1]) if c else None


def cleanup_old_checkpoints(exp_id, seed, best_path, latest_path):
    d = os.path.join(CONFIG["checkpoint_dir"], exp_id)
    keep = {os.path.basename(p) for p in [best_path, latest_path] if p}
    for f in os.listdir(d):
        if f.startswith(f"{exp_id}_seed{seed}_") and f not in keep:
            try: os.remove(os.path.join(d, f))
            except Exception: pass

print("Checkpoint utilities defined.")

Checkpoint utilities defined.


In [22]:
# =============================================================================
# SECTION 5 — CELL 5.3: train_one_epoch + validate   (identical to EXP-C)
# =============================================================================
from torch.amp import autocast, GradScaler

def train_one_epoch(model, loader, optimizer, scaler, exp_id):
    """One training epoch. exp_a/exp_b take the AMP path, exactly as in EXP-C."""
    model.train()
    _is_classical = exp_id in ("exp_a", "exp_b")
    total_loss, correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()

    for rgb, nir, labels in loader:
        rgb, nir, labels = rgb.cuda(), nir.cuda(), labels.cuda()
        optimizer.zero_grad()
        if _is_classical:
            with autocast('cuda'):
                logits = model(rgb, nir)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(rgb, nir)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for rgb, nir, labels in loader:
        rgb, nir, labels = rgb.cuda(), nir.cuda(), labels.cuda()
        logits = model(rgb, nir)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

print("train_one_epoch / validate defined.")

train_one_epoch / validate defined.


In [23]:
# =============================================================================
# SECTION 5 — CELL 5.4: train_model   (identical to EXP-C)
# =============================================================================
from tqdm import tqdm

def train_model(model, train_loader, val_loader, seed, exp_id, max_epochs=None):
    _max_ep = max_epochs or CONFIG["max_epochs"]
    _base_model = model.module if isinstance(model, torch.nn.DataParallel) else model

    optimizer  = build_optimizer(_base_model)
    scheduler  = build_scheduler(optimizer)
    scaler     = torch.amp.GradScaler('cuda')
    early_stop = EarlyStopping(CONFIG["early_stop_patience"], CONFIG["early_stop_min_delta"])
    training_log, _start_epoch = [], 1
    _best_val_loss, _best_ckpt_path = float("inf"), None

    _resume_path = get_latest_checkpoint(exp_id, seed)
    if _resume_path:
        print(f"  Resuming from {os.path.basename(_resume_path)}")
        _ckpt = load_checkpoint(_base_model, _resume_path, optimizer, scheduler)
        _start_epoch   = _ckpt["epoch"] + 1
        training_log   = _ckpt.get("training_log", [])
        _best_val_loss = min((e["val_loss"] for e in training_log), default=float("inf"))
        _best_ckpt_path = get_best_checkpoint(exp_id, seed)

    set_all_seeds(seed)

    for epoch in range(_start_epoch, _max_ep + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, exp_id)
        val_loss,   val_acc   = validate(model, val_loader)
        scheduler.step()

        training_log.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                             "val_loss": val_loss, "val_acc": val_acc})

        _is_best = val_loss < _best_val_loss
        if _is_best:
            _best_val_loss = val_loss
            if _best_ckpt_path and os.path.exists(_best_ckpt_path):
                try: os.remove(_best_ckpt_path)
                except Exception: pass
            _best_ckpt_path = save_checkpoint(model, optimizer, scheduler, epoch,
                                              val_loss, val_acc, seed, exp_id,
                                              training_log, is_best=True)

        _latest_path = save_checkpoint(model, optimizer, scheduler, epoch,
                                       val_loss, val_acc, seed, exp_id,
                                       training_log, is_best=False)
        cleanup_old_checkpoints(exp_id, seed, _best_ckpt_path, _latest_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{exp_id}|s{seed}] ep{epoch:03d}: "
                  f"train_loss={train_loss:.4f} acc={train_acc:.3f} | "
                  f"val_loss={val_loss:.4f} acc={val_acc:.3f}"
                  f"{' <- best' if _is_best else ''}")

        if early_stop.step(val_loss):
            print(f"  Early stopping at epoch {epoch}")
            break

    return training_log

print("train_model defined.")

train_model defined.


---
# Section 6 — Evaluation, Uncertainty, OOD

In [24]:
# =============================================================================
# SECTION 6 — CELL 6.1: Classification metrics   (identical to EXP-C)
# =============================================================================
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, cohen_kappa_score

@torch.no_grad()
def evaluate_classification(model, loader) -> dict:
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for rgb, nir, labels in loader:
        rgb, nir, labels = rgb.cuda(), nir.cuda(), labels.cuda()
        probs = torch.softmax(model(rgb, nir), dim=1)
        all_probs.append(probs.cpu())
        all_preds.append(probs.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())
    all_probs  = torch.cat(all_probs)
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    _p, _l = all_preds.numpy(), all_labels.numpy()
    return {
        "oa":           accuracy_score(_l, _p),
        "macro_f1":     f1_score(_l, _p, average="macro", zero_division=0),
        "per_class_f1": f1_score(_l, _p, average=None, zero_division=0).tolist(),
        "kappa":        cohen_kappa_score(_l, _p),
        "conf_mat":     confusion_matrix(_l, _p).tolist(),
        "probs":        all_probs,
        "labels":       all_labels,
    }

print("evaluate_classification defined.")

evaluate_classification defined.


In [25]:
# =============================================================================
# SECTION 6 — CELL 6.2: EnsembleInference   (identical to EXP-C)
# =============================================================================
class EnsembleInference:
    def __init__(self, models: list):
        self.models = [m.eval() for m in models]

    @torch.no_grad()
    def predict(self, rgb, nir) -> dict:
        member_probs = [torch.softmax(m(rgb, nir), dim=1).unsqueeze(0) for m in self.models]
        stack      = torch.cat(member_probs, dim=0)
        mean_probs = stack.mean(dim=0)
        variance   = stack.var(dim=0)
        entropy    = -(mean_probs * (mean_probs + 1e-10).log()).sum(dim=1)
        return {"mean_probs": mean_probs, "variance": variance, "entropy": entropy}

    @torch.no_grad()
    def evaluate_loader(self, loader) -> dict:
        all_probs, all_labels, all_entropy, all_var = [], [], [], []
        for rgb, nir, labels in loader:
            out = self.predict(rgb.cuda(), nir.cuda())
            all_probs.append(out["mean_probs"].cpu())
            all_labels.append(labels)
            all_entropy.append(out["entropy"].cpu())
            all_var.append(out["variance"].cpu())
        return {"probs": torch.cat(all_probs), "labels": torch.cat(all_labels),
                "entropy": torch.cat(all_entropy), "variance": torch.cat(all_var)}


def load_ensemble(exp_id: str, seeds: list) -> list:
    models = []
    for seed in seeds:
        ckpt_path = get_best_checkpoint(exp_id, seed)
        if ckpt_path is None:
            print(f"  WARNING: no checkpoint for {exp_id} seed={seed}")
            continue
        m = build_model(exp_id)
        load_checkpoint(m, ckpt_path); m.eval(); models.append(m)
        print(f"  Loaded {exp_id} seed={seed}: {os.path.basename(ckpt_path)}")
    return models

print("EnsembleInference / load_ensemble defined.")

EnsembleInference / load_ensemble defined.


In [26]:
# =============================================================================
# SECTION 6 — CELL 6.3: Uncertainty + OOD metrics   (identical to EXP-C)
# =============================================================================
def compute_ece(probs: torch.Tensor, labels: torch.Tensor) -> float:
    metric = torchmetrics.CalibrationError(
        task='multiclass', num_classes=CONFIG["n_classes"],
        n_bins=CONFIG["ece_n_bins"], norm='l1')
    return metric(probs, labels).item()


def evaluate_uncertainty(ensemble, loader) -> dict:
    out = ensemble.evaluate_loader(loader)
    probs, labels = out["probs"], out["labels"]
    _oa   = accuracy_score(labels.numpy(), probs.argmax(1).numpy())
    ece   = compute_ece(probs, labels)
    nll   = F.nll_loss((probs + 1e-10).log(), labels).item()
    _oh   = F.one_hot(labels, num_classes=CONFIG["n_classes"]).float()
    brier = ((probs - _oh) ** 2).sum(dim=1).mean().item()
    return {"oa": _oa, "ece": ece, "nll": nll, "brier": brier,
            "mean_entropy":  out["entropy"].mean().item(),
            "mean_variance": out["variance"].mean().item()}


def evaluate_ood(ensemble, loader, noise_levels: list) -> dict:
    """Additive Gaussian noise applied AFTER normalisation, to rgb and nir
    independently — identical to the EXP-C implementation."""
    results = {}
    for sigma in noise_levels:
        _all_probs, _all_labels = [], []
        for rgb, nir, labels in loader:
            rgb_n = (rgb + torch.randn_like(rgb) * sigma).cuda()
            nir_n = (nir + torch.randn_like(nir) * sigma).cuda()
            _all_probs.append(ensemble.predict(rgb_n, nir_n)["mean_probs"].cpu())
            _all_labels.append(labels)
        _probs, _labels = torch.cat(_all_probs), torch.cat(_all_labels)
        results[sigma] = {
            "oa":      accuracy_score(_labels.numpy(), _probs.argmax(1).numpy()),
            "ece":     compute_ece(_probs, _labels),
            "entropy": (-((_probs + 1e-10).log() * _probs).sum(1)).mean().item(),
            "nll":     F.nll_loss((_probs + 1e-10).log(), _labels).item(),
        }
    return results

# ── Sanity check ─────────────────────────────────────────────────────────────
_ece_uni = compute_ece(torch.ones(200, 10) / 10, torch.randint(0, 10, (200,)))
assert abs(_ece_uni) < 0.02, f"ECE sanity failed: {_ece_uni}"
print(f"ECE sanity (uniform prediction): {_ece_uni:.4f}")
print("evaluate_uncertainty / evaluate_ood defined.")

ECE sanity (uniform prediction): 0.0200
evaluate_uncertainty / evaluate_ood defined.


In [27]:
# =============================================================================
# SECTION 6 — CELL 6.4: Training sanity check (2 epochs, 128 samples)
# =============================================================================
from torch.utils.data import Subset

print("Running 2-epoch sanity check on EXP-A (128 samples)...")
set_all_seeds(42)
_tiny = Subset(train_dataset, range(128))
_tiny_loader = DataLoader(_tiny, batch_size=16, shuffle=True, num_workers=0)

_m_s = build_model("exp_a")
_log_s = train_model(_m_s, _tiny_loader, _tiny_loader, seed=42, exp_id="sanity", max_epochs=2)

assert not any(math.isnan(e["train_loss"]) for e in _log_s), "NaN in sanity loss"
assert os.path.exists(get_best_checkpoint("sanity", 42) or ""), "sanity checkpoint missing"

_m_s2 = build_model("exp_a")
load_checkpoint(_m_s2, get_best_checkpoint("sanity", 42))
_m_s.eval(); _m_s2.eval()
with torch.no_grad():
    _r_s, _n_s = torch.randn(4,3,64,64).cuda(), torch.randn(4,1,64,64).cuda()
    assert torch.allclose(_m_s(_r_s, _n_s), _m_s2(_r_s, _n_s), atol=1e-5), \
        "checkpoint reload not deterministic"

import shutil
shutil.rmtree(os.path.join(CONFIG["checkpoint_dir"], "sanity"), ignore_errors=True)
del _m_s, _m_s2, _tiny, _tiny_loader; torch.cuda.empty_cache(); gc.collect()
print("Section 6 sanity check PASSED")

Running 2-epoch sanity check on EXP-A (128 samples)...
  [sanity|s42] ep001: train_loss=2.5789 acc=0.281 | val_loss=476.5918 acc=0.125 <- best
Section 6 sanity check PASSED


---
# Section 7 — Training (EXP-A and EXP-B, all seeds)

In [28]:
# =============================================================================
# SECTION 7 — CELL 7.1: Results containers
# =============================================================================
RAW_RESULTS: dict = {}
ENSEMBLE_RESULTS: dict = {}
OOD_RESULTS: dict = {}
HESSIAN_RESULTS: dict = {}     # empty — Hessian is EXP-C/EXP-D only
RUN_TIMESTAMPS: dict = {}
RUN_DURATIONS: dict = {}

_metrics_path = os.path.join(CONFIG["exports_dir"], "metrics_ab.json")

if os.path.exists(_metrics_path) and not FORCE_RETRAIN:
    _loaded = json.load(open(_metrics_path))
    RAW_RESULTS      = _loaded.get("raw_results", {})
    ENSEMBLE_RESULTS = _loaded.get("ensemble_results", {})
    OOD_RESULTS      = _loaded.get("ood_results", {})
    print(f"Loaded existing metrics: {list(RAW_RESULTS)}")
else:
    print("Starting fresh (FORCE_RETRAIN=True or no metrics_ab.json).")

_wall_t0 = time.time()
print("=" * 60)
print(f"Experiments : {EXPERIMENT_IDS}")
print(f"Seeds       : {CONFIG['ensemble_seeds']}")
print(f"Max epochs  : {CONFIG['max_epochs']}  patience={CONFIG['early_stop_patience']}")
print(f"Total runs  : {len(EXPERIMENT_IDS) * len(CONFIG['ensemble_seeds'])}")
print("=" * 60)

Starting fresh (FORCE_RETRAIN=True or no metrics_ab.json).
Experiments : ['exp_a', 'exp_b']
Seeds       : [42, 2021, 7]
Max epochs  : 30  patience=7
Total runs  : 6


In [29]:
# =============================================================================
# SECTION 7 — CELL 7.2: Main training loop — EXP-A and EXP-B, all seeds
# =============================================================================
for EXP_ID in EXPERIMENT_IDS:
    print(f"\n{'#'*64}")
    print(f"# {EXP_ID.upper()}")
    print(f"{'#'*64}")

    if EXP_ID not in RAW_RESULTS:
        RAW_RESULTS[EXP_ID] = {}

    for seed in CONFIG["ensemble_seeds"]:
        _run_key = f"{EXP_ID}_seed{seed}"
        if (str(seed) in RAW_RESULTS[EXP_ID] or seed in RAW_RESULTS[EXP_ID]) and not FORCE_RETRAIN:
            print(f"[SKIP] {_run_key} — results already present.")
            continue

        print(f"\n{'='*60}")
        print(f"[RUN] {_run_key}")
        print(f"{'='*60}")

        # Same call order as EXP-C: seed -> build -> train
        set_all_seeds(seed)
        model = build_model(EXP_ID)
        _n_params = sum(p.numel() for p in model.parameters())
        print(f"  Model params: {_n_params:,}")

        _t0 = time.time()
        RUN_TIMESTAMPS[_run_key] = datetime.now().isoformat()

        log = train_model(model, train_loader, val_loader, seed=seed, exp_id=EXP_ID)

        _best = get_best_checkpoint(EXP_ID, seed)
        if _best:
            load_checkpoint(model, _best)
            print(f"  Loaded best checkpoint: {os.path.basename(_best)}")

        clf_results = evaluate_classification(model, test_loader)
        _clf_save = {k: v for k, v in clf_results.items() if k not in ("probs", "labels")}

        uq_results = evaluate_uncertainty(EnsembleInference([model]), test_loader)

        _dur = (time.time() - _t0) / 3600
        RUN_DURATIONS[_run_key] = _dur

        RAW_RESULTS[EXP_ID][seed] = {
            "classification":  _clf_save,
            "uncertainty":     uq_results,
            "training_log":    log,
            "n_params":        _n_params,
            "duration_h":      _dur,
            "epochs_trained":  len(log),
            "best_checkpoint": os.path.basename(_best) if _best else None,
        }

        print(f"  OA={clf_results['oa']:.4f}  F1={clf_results['macro_f1']:.4f}"
              f"  ECE={uq_results['ece']:.4f}  epochs={len(log)}  [{_dur:.2f}h]")

        save_json({"raw_results": RAW_RESULTS, "ensemble_results": ENSEMBLE_RESULTS,
                   "ood_results": OOD_RESULTS, "hessian_results": HESSIAN_RESULTS,
                   "dataset_manifest": DATASET_MANIFEST,
                   "split_fingerprint": SPLIT_FINGERPRINT,
                   "stats_fingerprint": STATS_FINGERPRINT,
                   "duplicate_report": DUPLICATE_REPORT,
                   "config_hash": CONFIG_HASH},
                  _metrics_path)

        del model; torch.cuda.empty_cache(); gc.collect()

print(f"\nAll training runs complete. Wall time: {(time.time()-_wall_t0)/3600:.2f} h")


################################################################
# EXP_A
################################################################

[RUN] exp_a_seed42
  Model params: 11,173,962
  [exp_a|s42] ep001: train_loss=0.9138 acc=0.682 | val_loss=1.0751 acc=0.647 <- best
  [exp_a|s42] ep005: train_loss=0.3689 acc=0.878 | val_loss=0.4534 acc=0.855 <- best
  [exp_a|s42] ep010: train_loss=0.2255 acc=0.923 | val_loss=0.9893 acc=0.769
  [exp_a|s42] ep015: train_loss=0.1616 acc=0.946 | val_loss=0.1860 acc=0.942
  [exp_a|s42] ep020: train_loss=0.0983 acc=0.967 | val_loss=0.1026 acc=0.962
  [exp_a|s42] ep025: train_loss=0.0590 acc=0.980 | val_loss=0.0628 acc=0.978 <- best
  [exp_a|s42] ep030: train_loss=0.0424 acc=0.986 | val_loss=0.0572 acc=0.980 <- best
  Loaded best checkpoint: exp_a_seed42_epoch030_valAcc0.9800_best.pt


/tmp/ipykernel_23/1579198794.py:13: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  variance   = stack.var(dim=0)


  OA=0.9797  F1=0.9785  ECE=0.0079  epochs=30  [0.27h]

[RUN] exp_a_seed2021
  Model params: 11,173,962
  [exp_a|s2021] ep001: train_loss=0.9172 acc=0.679 | val_loss=0.7471 acc=0.755 <- best
  [exp_a|s2021] ep005: train_loss=0.3857 acc=0.870 | val_loss=0.6747 acc=0.802
  [exp_a|s2021] ep010: train_loss=0.2535 acc=0.915 | val_loss=0.3431 acc=0.883 <- best
  [exp_a|s2021] ep015: train_loss=0.1776 acc=0.940 | val_loss=0.1592 acc=0.945 <- best
  [exp_a|s2021] ep020: train_loss=0.1208 acc=0.959 | val_loss=0.1130 acc=0.964 <- best
  [exp_a|s2021] ep025: train_loss=0.0728 acc=0.975 | val_loss=0.0879 acc=0.972
  [exp_a|s2021] ep030: train_loss=0.0514 acc=0.984 | val_loss=0.0626 acc=0.979 <- best
  Loaded best checkpoint: exp_a_seed2021_epoch030_valAcc0.9795_best.pt


/tmp/ipykernel_23/1579198794.py:13: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  variance   = stack.var(dim=0)


  OA=0.9756  F1=0.9741  ECE=0.0084  epochs=30  [0.26h]

[RUN] exp_a_seed7
  Model params: 11,173,962
  [exp_a|s7] ep001: train_loss=0.9028 acc=0.683 | val_loss=1.0154 acc=0.659 <- best
  [exp_a|s7] ep005: train_loss=0.3436 acc=0.885 | val_loss=0.5520 acc=0.811
  [exp_a|s7] ep010: train_loss=0.2479 acc=0.918 | val_loss=0.3495 acc=0.893
  [exp_a|s7] ep015: train_loss=0.1574 acc=0.946 | val_loss=0.1477 acc=0.951 <- best
  [exp_a|s7] ep020: train_loss=0.1047 acc=0.965 | val_loss=0.0926 acc=0.967 <- best
  [exp_a|s7] ep025: train_loss=0.0623 acc=0.978 | val_loss=0.0750 acc=0.974
  [exp_a|s7] ep030: train_loss=0.0467 acc=0.984 | val_loss=0.0543 acc=0.980 <- best
  Loaded best checkpoint: exp_a_seed7_epoch030_valAcc0.9802_best.pt


...

  OA=0.9850  F1=0.9842  ECE=0.0067  epochs=30  [0.26h]

[RUN] exp_b_seed2021
  Model params: 11,174,538
  [exp_b|s2021] ep001: train_loss=0.6776 acc=0.773 | val_loss=0.4160 acc=0.854 <- best
  [exp_b|s2021] ep005: train_loss=0.2444 acc=0.917 | val_loss=0.4408 acc=0.882
  [exp_b|s2021] ep010: train_loss=0.1515 acc=0.949 | val_loss=0.1487 acc=0.948 <- best
  [exp_b|s2021] ep015: train_loss=0.1033 acc=0.967 | val_loss=0.1048 acc=0.966
  [exp_b|s2021] ep020: train_loss=0.0679 acc=0.977 | val_loss=0.0646 acc=0.980 <- best
  [exp_b|s2021] ep025: train_loss=0.0379 acc=0.987 | val_loss=0.0626 acc=0.979
  [exp_b|s2021] ep030: train_loss=0.0292 acc=0.991 | val_loss=0.0506 acc=0.984 <- best
  Loaded best checkpoint: exp_b_seed2021_epoch030_valAcc0.9838_best.pt


/tmp/ipykernel_23/1579198794.py:13: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  variance   = stack.var(dim=0)


  OA=0.9867  F1=0.9857  ECE=0.0043  epochs=30  [0.26h]

[RUN] exp_b_seed7
  Model params: 11,174,538
  [exp_b|s7] ep001: train_loss=0.6882 acc=0.771 | val_loss=0.7409 acc=0.779 <- best
  [exp_b|s7] ep005: train_loss=0.2367 acc=0.922 | val_loss=0.3000 acc=0.898
  [exp_b|s7] ep010: train_loss=0.1589 acc=0.947 | val_loss=0.1981 acc=0.933
  [exp_b|s7] ep015: train_loss=0.1047 acc=0.965 | val_loss=0.0959 acc=0.968 <- best
  [exp_b|s7] ep020: train_loss=0.0651 acc=0.978 | val_loss=0.0743 acc=0.974 <- best
  [exp_b|s7] ep025: train_loss=0.0391 acc=0.987 | val_loss=0.0545 acc=0.982
  [exp_b|s7] ep030: train_loss=0.0294 acc=0.990 | val_loss=0.0475 acc=0.984
  Loaded best checkpoint: exp_b_seed7_epoch029_valAcc0.9841_best.pt


/tmp/ipykernel_23/1579198794.py:13: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  variance   = stack.var(dim=0)


  OA=0.9872  F1=0.9861  ECE=0.0047  epochs=30  [0.26h]

All training runs complete. Wall time: 1.59 h


In [30]:
# =============================================================================
# SECTION 7 — CELL 7.3: Ensemble + OOD evaluation
# =============================================================================
for EXP_ID in EXPERIMENT_IDS:
    print(f"\nEnsemble evaluation: {EXP_ID}")
    _ens_models = load_ensemble(EXP_ID, CONFIG["ensemble_seeds"])
    if len(_ens_models) < 2:
        print(f"  Only {len(_ens_models)} checkpoints — skipping ensemble UQ.")
        continue

    ensemble = EnsembleInference(_ens_models)
    ENSEMBLE_RESULTS[EXP_ID] = evaluate_uncertainty(ensemble, test_loader)
    OOD_RESULTS[EXP_ID]      = evaluate_ood(ensemble, test_loader, CONFIG["ood_noise_levels"])

    _e = ENSEMBLE_RESULTS[EXP_ID]
    print(f"  Ensemble OA={_e['oa']:.4f}  ECE={_e['ece']:.4f}"
          f"  NLL={_e['nll']:.4f}  Brier={_e['brier']:.4f}")
    print("  OOD:  " + "  ".join(
        f"s={s}:{OOD_RESULTS[EXP_ID][s]['oa']*100:.1f}%" for s in CONFIG["ood_noise_levels"]))

    del _ens_models, ensemble; torch.cuda.empty_cache(); gc.collect()

save_json({"raw_results": RAW_RESULTS, "ensemble_results": ENSEMBLE_RESULTS,
           "ood_results": OOD_RESULTS, "hessian_results": HESSIAN_RESULTS,
           "dataset_manifest": DATASET_MANIFEST,
           "split_fingerprint": SPLIT_FINGERPRINT,
           "stats_fingerprint": STATS_FINGERPRINT,
           "duplicate_report": DUPLICATE_REPORT,
           "config_hash": CONFIG_HASH},
          _metrics_path)
print(f"\nResults saved to {_metrics_path}")


Ensemble evaluation: exp_a
  Loaded exp_a seed=42: exp_a_seed42_epoch030_valAcc0.9800_best.pt
  Loaded exp_a seed=2021: exp_a_seed2021_epoch030_valAcc0.9795_best.pt
  Loaded exp_a seed=7: exp_a_seed7_epoch030_valAcc0.9802_best.pt
  Ensemble OA=0.9809  ECE=0.0038  NLL=0.0602  Brier=0.0301
  OOD:  s=0.0:98.1%  s=0.01:98.0%  s=0.05:88.2%  s=0.1:66.4%  s=0.2:28.3%

Ensemble evaluation: exp_b
  Loaded exp_b seed=42: exp_b_seed42_epoch030_valAcc0.9855_best.pt
  Loaded exp_b seed=2021: exp_b_seed2021_epoch030_valAcc0.9838_best.pt
  Loaded exp_b seed=7: exp_b_seed7_epoch029_valAcc0.9841_best.pt
  Ensemble OA=0.9891  ECE=0.0046  NLL=0.0358  Brier=0.0184
  OOD:  s=0.0:98.9%  s=0.01:98.9%  s=0.05:98.6%  s=0.1:96.2%  s=0.2:73.5%

Results saved to /kaggle/working/Quantum-Geoscience/exports/metrics_ab.json


---
# Section 8 — Summary and Export

In [31]:
# =============================================================================
# SECTION 8 — CELL 8.1: Summary + protocol-match report
# =============================================================================
from statistics import mean, stdev

print("=" * 72)
print("EXP-A / EXP-B RE-RUN SUMMARY")
print("=" * 72)

print(f"\nDataset      : {DATASET_MANIFEST['total_images']} images "
      f"({'canonical' if DATASET_MANIFEST['is_canonical_eurosat'] else 'NON-canonical'})")
if DATASET_MANIFEST["deviations_from_canonical"]:
    print(f"  deviations : {DATASET_MANIFEST['deviations_from_canonical']}")
print(f"Split        : train={CONFIG['n_train']} val={CONFIG['n_val']} test={CONFIG['n_test']}")
print(f"  test md5   : {SPLIT_FINGERPRINT['test_md5']}")
print(f"  matches quantum arm test composition: YES (asserted in Cell 3.5b)")
if DUPLICATE_REPORT.get("ran"):
    print(f"Duplicates   : SeaLake unique={DUPLICATE_REPORT['unique_images']}"
          f"/{DUPLICATE_REPORT['file_count']}"
          f"  redundant={DUPLICATE_REPORT['redundant_files']}"
          f"  leaky_groups={DUPLICATE_REPORT['groups_straddling_train_and_heldout']}")

print(f"\n{'-'*72}")
print(f"{'exp':<8}{'seed':>7}{'epochs':>9}{'OA %':>9}{'macroF1':>10}{'ECE':>9}{'hours':>8}")
print(f"{'-'*72}")
for EXP_ID in EXPERIMENT_IDS:
    _oas = []
    for seed in CONFIG["ensemble_seeds"]:
        _r = RAW_RESULTS.get(EXP_ID, {}).get(seed) or RAW_RESULTS.get(EXP_ID, {}).get(str(seed))
        if not _r: continue
        _oas.append(_r["classification"]["oa"])
        print(f"{EXP_ID:<8}{seed:>7}{_r.get('epochs_trained','?'):>9}"
              f"{_r['classification']['oa']*100:>9.2f}"
              f"{_r['classification']['macro_f1']:>10.4f}"
              f"{_r['uncertainty']['ece']:>9.5f}"
              f"{_r['duration_h']:>8.2f}")
    if len(_oas) > 1:
        print(f"{'':<8}{'mean':>7}{'':>9}{mean(_oas)*100:>9.2f}"
              f"{'  +/- ' + format(stdev(_oas)*100, '.2f'):>10}")
    print(f"{'-'*72}")

print(f"\n{'exp':<8}{'ENSEMBLE OA %':>16}{'ECE':>10}{'NLL':>10}{'Brier':>10}{'epi var':>11}")
for EXP_ID in EXPERIMENT_IDS:
    _e = ENSEMBLE_RESULTS.get(EXP_ID)
    if _e:
        print(f"{EXP_ID:<8}{_e['oa']*100:>16.2f}{_e['ece']:>10.5f}"
              f"{_e['nll']:>10.4f}{_e['brier']:>10.4f}{_e['mean_variance']:>11.6f}")

print("\n" + "=" * 72)
print("PROTOCOL MATCH vs EXP-C")
print("=" * 72)
for _k, _v in [("dataset", DATASET_SLUG), ("split_seed", 42), ("batch_size", CONFIG['batch_size']),
               ("max_epochs (all seeds)", CONFIG['max_epochs']),
               ("patience", CONFIG['early_stop_patience']),
               ("lr / wd", f"{CONFIG['lr']} / {CONFIG['weight_decay_classical']}"),
               ("scheduler", f"CosineAnnealingLR T_max={CONFIG['cosine_T_max']}"),
               ("seeds", CONFIG['ensemble_seeds']), ("ECE bins", CONFIG['ece_n_bins']),
               ("OOD sigmas", CONFIG['ood_noise_levels'])]:
    print(f"  {_k:<26} {_v}")
print("\nEXP-A previously ran 50 epochs on seed 42 and 30 on the others.")
print("All seeds now run the same budget. That inconsistency is resolved.")

EXP-A / EXP-B RE-RUN SUMMARY

Dataset      : 27597 images (NON-canonical)
  deviations : {'SeaLake': 597}
Split        : train=19317 val=4140 test=4140
  test md5   : 5d5ff68d7e75b127
  matches quantum arm test composition: YES (asserted in Cell 3.5b)
Duplicates   : SeaLake unique=3597/3597  redundant=0  leaky_groups=0

------------------------------------------------------------------------
exp        seed   epochs     OA %   macroF1      ECE   hours
------------------------------------------------------------------------
exp_a        42       30    97.97    0.9785  0.00790    0.27
exp_a      2021       30    97.56    0.9741  0.00843    0.26
exp_a         7       30    97.73    0.9757  0.00634    0.26
           mean             97.75  +/- 0.21
------------------------------------------------------------------------
exp_b        42       30    98.50    0.9842  0.00673    0.26
exp_b      2021       30    98.67    0.9857  0.00428    0.26
exp_b         7       30    98.72    0.9861  0.00

In [32]:
# =============================================================================
# SECTION 8 — CELL 8.2: Export for merging with quantum results
# =============================================================================
import shutil, glob

_out_dir = "/kaggle/working/exp_ab_output"
os.makedirs(_out_dir, exist_ok=True)

shutil.copy2(_metrics_path, os.path.join(_out_dir, "metrics_ab.json"))
shutil.copy2(os.path.join(CONFIG["exports_dir"], "configuration_ab.json"),
             os.path.join(_out_dir, "configuration_ab.json"))

for _e in EXPERIMENT_IDS:
    _src = os.path.join(CONFIG["checkpoint_dir"], _e)
    _dst = os.path.join(_out_dir, "checkpoints", _e)
    os.makedirs(_dst, exist_ok=True)
    for _f in glob.glob(os.path.join(_src, "*_best*.pt")):
        shutil.copy2(_f, _dst)

np.save(os.path.join(_out_dir, "train_indices.npy"), _train_idx)
np.save(os.path.join(_out_dir, "val_indices.npy"),   _val_idx)
np.save(os.path.join(_out_dir, "test_indices.npy"),  _test_idx)
np.save(os.path.join(_out_dir, "channel_mean.npy"),  CHANNEL_MEAN)
np.save(os.path.join(_out_dir, "channel_std.npy"),   CHANNEL_STD)

print(f"Exported to {_out_dir}:")
for _r, _d, _fs in os.walk(_out_dir):
    for _f in _fs:
        _p = os.path.join(_r, _f)
        print(f"  {os.path.relpath(_p, _out_dir):<44} {os.path.getsize(_p)/1e6:>8.2f} MB")

print("\nNEXT STEPS")
print("  1. Save Version -> Save & Run All, then download exp_ab_output/.")
print("  2. Merge metrics_ab.json into your combined metrics file:")
print("       merged['raw_results']['exp_a'] = ab['raw_results']['exp_a']   # same for exp_b")
print("       merged['ensemble_results'] / ['ood_results'] likewise")
print("  3. Confirm every experiment now reports test n = "
      f"{CONFIG['n_test']} before running any comparison.")

Exported to /kaggle/working/exp_ab_output:
  val_indices.npy                                  0.03 MB
  configuration_ab.json                            0.00 MB
  metrics_ab.json                                  0.06 MB
  train_indices.npy                                0.15 MB
  test_indices.npy                                 0.03 MB
  channel_std.npy                                  0.00 MB
  channel_mean.npy                                 0.00 MB
  checkpoints/exp_a/exp_a_seed2021_epoch030_valAcc0.9795_best.pt   134.25 MB
  checkpoints/exp_a/exp_a_seed42_epoch030_valAcc0.9800_best.pt   134.25 MB
  checkpoints/exp_a/exp_a_seed7_epoch030_valAcc0.9802_best.pt   134.25 MB
  checkpoints/exp_b/exp_b_seed7_epoch029_valAcc0.9841_best.pt   134.25 MB
  checkpoints/exp_b/exp_b_seed2021_epoch030_valAcc0.9838_best.pt   134.25 MB
  checkpoints/exp_b/exp_b_seed42_epoch030_valAcc0.9855_best.pt   134.25 MB

NEXT STEPS
  1. Save Version -> Save & Run All, then download exp_ab_output/.
  2. Merge me